# Tourism-Yearly Comprehensive Sweep Analysis

**Date:** 2026-04-06  
**Dataset:** Tourism-Yearly (H=4, L=20)  
**Scope:** 112 configurations x 10 runs each = 1,120 total runs  
**Data:** `experiments/results/tourism/comprehensive_sweep_tourism_results.csv`

This notebook analyzes the comprehensive Tourism-Yearly sweep covering 14 architecture categories:
paper baselines, TrendWavelet (RootBlock & AE/AELG variants), alternating stacks, Generic/BottleneckGeneric (AE/AELG),
and TrendWaveletGeneric. Key questions:

1. Does any config beat the known Tourism-Yearly SOTA (SMAPE 20.864)?
2. Which architecture family is best for short-horizon Tourism forecasting?
3. How do active_g, depth, skip connections, and latent dim interact?
4. What is the most parameter-efficient competitive config?

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 6)
matplotlib.rcParams['font.size'] = 11

df = pd.read_csv('../../results/tourism/comprehensive_sweep_tourism_results.csv')
df = df[df['period'] == 'Tourism-Yearly'].copy()
print(f'Loaded {len(df)} runs across {df["config_name"].nunique()} configs')
print(f'Categories: {sorted(df["category"].unique())}')

## 1. Overall Rankings

Which configurations achieve the lowest SMAPE on Tourism-Yearly?

In [ ]:
agg = df.groupby('config_name').agg(
    mean_smape=('smape', 'mean'),
    std_smape=('smape', 'std'),
    median_smape=('smape', 'median'),
    min_smape=('smape', 'min'),
    max_smape=('smape', 'max'),
    mean_mase=('mase', 'mean'),
    n_params=('n_params', 'first'),
    category=('category', 'first'),
    active_g=('active_g_cfg', 'first'),
).sort_values('mean_smape')

best = agg['mean_smape'].iloc[0]
display_df = agg.head(20).copy()
display_df['delta_pct'] = (display_df['mean_smape'] - best) / best * 100
display_df['params_M'] = display_df['n_params'] / 1e6
display_df[['mean_smape','std_smape','median_smape','mean_mase','params_M','category','delta_pct']].round(3)

**Key finding:** The sweep winner is `TW_10s_td3_bdeq_db3` (non-AE TrendWavelet, 10 stacks, td3, basis_dim=eq_fcast, db3 wavelet) at SMAPE 21.773. This is a solid configuration but does **not** beat the known Tourism SOTA of 20.864 (TrendWaveletAELG with coif3) or the GenericAE SOTA of 20.526 from the resnet skip study.

The top 10 is remarkably diverse: non-AE TrendWavelet, BottleneckGenericAELG, GenericAELG, TrendWaveletAELG, BottleneckGeneric, BottleneckGenericAE, and GenericAE all appear. The active_g=forecast setting dominates the top spots.

In [ ]:
# Visualize top 30 configs
top30 = agg.head(30)
fig, ax = plt.subplots(figsize=(16, 8))
colors = {'trendwavelet_rb': '#2196F3', 'trendwavelet_td': '#1565C0',
          'trendwaveletaelg': '#4CAF50', 'trendwaveletae': '#8BC34A',
          'bottleneckgenericaelg': '#FF9800', 'bottleneckgenericae': '#FFB74D',
          'bottleneckgeneric': '#FF5722', 'genericaelg_skip': '#9C27B0',
          'genericae': '#E91E63', 'paper_baseline': '#607D8B',
          'alt_aelg': '#00BCD4', 'alt_ae': '#009688',
          'alt_trend_wavelet_rb': '#3F51B5', 'trendwaveletgenericaelg': '#795548'}

bar_colors = [colors.get(top30.loc[c, 'category'], '#999') for c in top30.index]
bars = ax.barh(range(len(top30)), top30['mean_smape'], xerr=top30['std_smape'],
               color=bar_colors, alpha=0.8, capsize=3)
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(top30.index, fontsize=8)
ax.set_xlabel('Mean SMAPE')
ax.set_title('Top 30 Tourism-Yearly Configs by Mean SMAPE')
ax.axvline(20.864, color='red', linestyle='--', alpha=0.7, label='Known SOTA (20.864)')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

## 2. Bimodal Convergence Failures

Seven configs show extremely high variance (std > 5), indicating bimodal convergence where 1-2 out of 10 seeds fail catastrophically with SMAPE ~66-69. This is the same bimodal pattern seen on M4-Yearly.

In [ ]:
bimodal = agg[agg['std_smape'] > 5].sort_values('std_smape', ascending=False)
print(f'{len(bimodal)} bimodal configs detected:\n')
for name, row in bimodal.iterrows():
    runs = df[df['config_name'] == name]['smape']
    good = runs[runs < 40]
    bad = runs[runs >= 40]
    print(f'  {name:<50} full={row["mean_smape"]:.1f} robust={good.mean():.3f} '
          f'({len(good)}/10 good, {len(bad)} failures)')

print(f'\nAll bimodal configs use active_g=False. Pattern: ag0 + Generic-family blocks '
      f'or small latent dim (ld8) -> 10% failure rate on Tourism.')

## 3. Active_g is the Single Most Impactful Setting

On Tourism-Yearly, `active_g=forecast` dramatically outperforms `active_g=False` across 38 paired configs. This is the opposite of Weather-96 where agf is catastrophic. Tourism confirms: **active_g is a dataset-level setting**.

In [ ]:
# Paired active_g comparison
ag0_means, agf_means, labels = [], [], []
for c in agg.index:
    if '_ag0' in c and '_agf' not in c:
        c_agf = c.replace('_ag0', '_agf')
        if c_agf in agg.index:
            ag0_means.append(agg.loc[c, 'mean_smape'])
            agf_means.append(agg.loc[c_agf, 'mean_smape'])
            labels.append(c.replace('_ag0', ''))

ag0_arr = np.array(ag0_means)
agf_arr = np.array(agf_means)
stat, pval = stats.wilcoxon(ag0_arr, agf_arr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
ax1.scatter(ag0_arr, agf_arr, alpha=0.7, s=50)
lims = [min(ag0_arr.min(), agf_arr.min())-0.5, max(ag0_arr.max(), agf_arr.max())+0.5]
ax1.plot(lims, lims, 'k--', alpha=0.3)
ax1.set_xlabel('ag0 SMAPE')
ax1.set_ylabel('agf SMAPE')
ax1.set_title(f'active_g=False vs forecast (Wilcoxon p={pval:.4f})\nagf wins {(agf_arr<ag0_arr).sum()}/'
              f'{len(ag0_arr)}, mean delta={np.mean(ag0_arr-agf_arr):+.3f}')

# Delta distribution
deltas = ag0_arr - agf_arr
ax2.hist(deltas, bins=15, alpha=0.7, edgecolor='black')
ax2.axvline(0, color='red', linestyle='--')
ax2.set_xlabel('SMAPE delta (ag0 - agf, positive = agf better)')
ax2.set_ylabel('Count')
ax2.set_title(f'Distribution of active_g effect (mean={deltas.mean():+.3f})')
plt.tight_layout()
plt.show()

# Note: the huge deltas are from bimodal configs where ag0 has failures but agf doesn't
print(f'\nFiltering to stable pairs only (both std < 5):')
stable_ag0, stable_agf = [], []
for c in agg.index:
    if '_ag0' in c and '_agf' not in c:
        c_agf = c.replace('_ag0', '_agf')
        if c_agf in agg.index and agg.loc[c, 'std_smape'] < 5 and agg.loc[c_agf, 'std_smape'] < 5:
            stable_ag0.append(agg.loc[c, 'mean_smape'])
            stable_agf.append(agg.loc[c_agf, 'mean_smape'])

s0 = np.array(stable_ag0)
sf = np.array(stable_agf)
stat2, pval2 = stats.wilcoxon(s0, sf)
print(f'  {len(s0)} stable pairs: ag0 mean={s0.mean():.3f}, agf mean={sf.mean():.3f}')
print(f'  agf wins: {(sf<s0).sum()}/{len(s0)}, Wilcoxon p={pval2:.4f}')
print(f'  Mean delta: {(s0-sf).mean():+.3f} SMAPE (agf better by this much)')

## 4. Depth: 10 Stacks Dominate on Tourism

Tourism-Yearly has very short forecast horizon (H=4). Does depth help?

In [ ]:
pairs = []
for c in agg.index:
    if '_10s_' in c:
        c30 = c.replace('_10s_', '_30s_')
        if c30 in agg.index:
            pairs.append((c, c30, agg.loc[c, 'mean_smape'], agg.loc[c30, 'mean_smape']))

s10 = np.array([p[2] for p in pairs])
s30 = np.array([p[3] for p in pairs])
stat, pval = stats.wilcoxon(s10, s30)

print(f'{len(pairs)} depth pairs. 10-stack wins: {(s10<s30).sum()}, 30-stack wins: {(s30<s10).sum()}')
print(f'Mean 10s: {s10.mean():.3f}, Mean 30s: {s30.mean():.3f}, delta: {(s30-s10).mean():+.3f}')
print(f'Wilcoxon p={pval:.6f}')
print(f'\nConclusion: 10 stacks is STRONGLY preferred for Tourism-Yearly (p<0.001). '
      f'30 stacks adds 0.9 SMAPE on average with 3x parameters.')

## 5. Skip Connections Hurt on Tourism

Consistent with prior resnet skip study findings: skip connections are detrimental on Tourism.

In [ ]:
skip_pairs = []
for c in agg.index:
    if '_sd5' in c:
        c_no = c.replace('_sd5', '')
        if c_no in agg.index:
            skip_pairs.append((c_no, c))

no_skip = np.array([agg.loc[p[0], 'mean_smape'] for p in skip_pairs])
with_skip = np.array([agg.loc[p[1], 'mean_smape'] for p in skip_pairs])

print(f'{len(skip_pairs)} skip pairs. Skip helps: {(with_skip < no_skip).sum()}, '
      f'Skip hurts: {(with_skip > no_skip).sum()}')
print(f'Mean no-skip: {no_skip.mean():.3f}, Mean skip: {with_skip.mean():.3f}')
for p in skip_pairs:
    m_no = agg.loc[p[0], 'mean_smape']
    m_sk = agg.loc[p[1], 'mean_smape']
    print(f'  {p[0]:<50} {m_no:.3f} -> {p[1]:<50} {m_sk:.3f} ({m_sk-m_no:+.3f})')

## 6. Parameter Efficiency: Pareto Frontier

Which configs achieve the best SMAPE per parameter?

In [ ]:
stable = agg[agg['std_smape'] < 5].copy()

fig, ax = plt.subplots(figsize=(14, 8))
for cat in sorted(stable['category'].unique()):
    mask = stable['category'] == cat
    ax.scatter(stable.loc[mask, 'n_params']/1e6, stable.loc[mask, 'mean_smape'],
              label=cat, alpha=0.7, s=60,
              color=colors.get(cat, '#999'))

# Pareto frontier
sorted_by_params = stable.sort_values('n_params')
pareto_names = []
best_so_far = float('inf')
for name, row in sorted_by_params.iterrows():
    if row['mean_smape'] < best_so_far:
        pareto_names.append(name)
        best_so_far = row['mean_smape']

pareto_df = stable.loc[pareto_names]
ax.plot(pareto_df['n_params']/1e6, pareto_df['mean_smape'], 'r--', alpha=0.5, linewidth=2)
for name in pareto_names:
    ax.annotate(name, (stable.loc[name, 'n_params']/1e6, stable.loc[name, 'mean_smape']),
               fontsize=6, rotation=15, alpha=0.8)

ax.axhline(20.864, color='red', linestyle=':', alpha=0.5, label='Known SOTA (20.864)')
ax.set_xlabel('Parameters (M)')
ax.set_ylabel('Mean SMAPE')
ax.set_title('Tourism-Yearly: SMAPE vs Parameter Count')
ax.set_xscale('log')
ax.legend(fontsize=7, ncol=3, loc='upper right')
plt.tight_layout()
plt.show()

print('Pareto-optimal configs:')
for name in pareto_names:
    row = stable.loc[name]
    print(f'  {name:<50} SMAPE={row["mean_smape"]:.3f} Params={row["n_params"]:>12,.0f} '
          f'({row["category"]})')

## 7. Wavelet Family Matters for Non-AE, Not for AE

Does wavelet choice make a difference?

In [ ]:
# Non-AE TrendWavelet at 10 stacks, td3, bdeq
wav_rb = {}
for wav in ['db3', 'coif2', 'sym10', 'haar']:
    c = f'TW_10s_td3_bdeq_{wav}'
    if c in agg.index:
        wav_rb[wav] = df[df['config_name'] == c]['smape'].values

if len(wav_rb) >= 3:
    kw_stat, kw_p = stats.kruskal(*wav_rb.values())
    print(f'Non-AE TW_10s_td3_bdeq_* wavelet effect: KW H={kw_stat:.2f}, p={kw_p:.4f}')
    for w, v in sorted(wav_rb.items(), key=lambda x: x[1].mean()):
        print(f'  {w}: {v.mean():.3f} +/- {v.std():.3f}')

# AELG at 10 stacks, ld16
print()
wav_aelg = {}
for wav in ['db3', 'coif2', 'sym10', 'haar']:
    c = f'TWAELG_10s_ld16_{wav}_ag0'
    if c in agg.index:
        wav_aelg[wav] = df[df['config_name'] == c]['smape'].values

if len(wav_aelg) >= 3:
    kw_stat, kw_p = stats.kruskal(*wav_aelg.values())
    print(f'AELG TWAELG_10s_ld16_*_ag0 wavelet effect: KW H={kw_stat:.2f}, p={kw_p:.4f}')
    for w, v in sorted(wav_aelg.items(), key=lambda x: x[1].mean()):
        print(f'  {w}: {v.mean():.3f} +/- {v.std():.3f}')

print('\nConclusion: Wavelet family has marginal effect on non-AE (~0.66 SMAPE spread) '
      'and borderline effect on AELG (~0.97 spread). db3 and coif2 are generally safest. '
      'Haar is worst for AELG (consistent with prior Tourism findings).')

## 8. Architecture Category Rankings

Comparing broad architecture families, filtering out bimodal failures for fair comparison.

In [ ]:
stable_df = df[df.groupby('config_name')['smape'].transform('std') < 5]

cat_agg = stable_df.groupby('category').agg(
    mean_smape=('smape', 'mean'),
    median_smape=('smape', 'median'),
    std_smape=('smape', 'std'),
    n_configs=('config_name', 'nunique'),
    mean_params=('n_params', 'mean'),
).sort_values('mean_smape')

fig, ax = plt.subplots(figsize=(12, 6))
bar_c = [colors.get(c, '#999') for c in cat_agg.index]
ax.barh(range(len(cat_agg)), cat_agg['mean_smape'], color=bar_c, alpha=0.8)
ax.set_yticks(range(len(cat_agg)))
ax.set_yticklabels([f"{c} ({int(cat_agg.loc[c, 'n_configs'])}cfg, {cat_agg.loc[c, 'mean_params']/1e6:.1f}M)" 
                    for c in cat_agg.index], fontsize=9)
ax.set_xlabel('Mean SMAPE')
ax.set_title('Category Rankings (stable configs only)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

cat_agg.round(3)

## 9. Statistical Significance: Top Configs

Can we statistically distinguish the top configurations from each other?

In [ ]:
top10_names = list(agg.head(10).index)
best_name = top10_names[0]
best_runs = df[df['config_name'] == best_name]['smape'].values

print(f'Reference: {best_name} (mean={best_runs.mean():.3f})\n')
print(f'{"Config":<50} {"Mean":>8} {"MWU-p":>10} {"Cohen-d":>9}')
print('-'*80)
for name in top10_names[1:]:
    runs = df[df['config_name'] == name]['smape'].values
    _, mwu_p = stats.mannwhitneyu(best_runs, runs, alternative='two-sided')
    pooled_std = np.sqrt((best_runs.std()**2 + runs.std()**2) / 2)
    d = (runs.mean() - best_runs.mean()) / pooled_std if pooled_std > 0 else 0
    sig = '***' if mwu_p < 0.001 else '**' if mwu_p < 0.01 else '*' if mwu_p < 0.05 else 'ns'
    print(f'{name:<50} {runs.mean():>8.3f} {mwu_p:>10.4f} {d:>+9.3f} {sig}')

print('\nNone of the top 9 configs are statistically distinguishable from #1 at p<0.05. '
      'The spread is only 0.30 SMAPE (21.77-22.08), well within noise.')

## 10. Comparison with Known Tourism SOTA

The known Tourism-Yearly SOTA is 20.864 (TrendWaveletAELG coif3_eq_bcast_td3_ld16) from a dedicated study. The best config in this sweep is 21.773, a +4.3% gap. Why?

The prior SOTA used `coif3` wavelet with `eq_bcast` basis dimension, neither of which was tested in this sweep (this sweep uses coif2/db3/haar/sym10 with eq_fcast/2x_eq). The prior SOTA was also specifically tuned for Tourism-Yearly with the right combination of parameters.

This confirms that the comprehensive sweep's grid, while broad, does not cover the specific configuration space that produced the Tourism SOTA.

In [ ]:
# Best individual runs
best_singles = df.nsmallest(15, 'smape')[['config_name', 'smape', 'seed', 'n_params']]
print('Best 15 individual runs:')
for _, row in best_singles.iterrows():
    marker = '*' if row['smape'] < 20.864 else ' '
    print(f'  {marker} {row["config_name"]:<50} SMAPE={row["smape"]:.3f} '
          f'params={row["n_params"]:,.0f}')

print(f'\n5 individual runs beat the SOTA threshold of 20.864, but none are stable '
      f'enough across 10 seeds to make a SOTA claim.')

## 11. Recommendations

### Current Best Configuration (from this sweep)

**Winner: `TW_10s_td3_bdeq_db3`** (SMAPE 21.773, 2.0M params)  
But this does NOT beat known SOTA (20.864). Confidence: **medium** -- tied with ~8 configs within noise.

**Most parameter-efficient: `TWAELG_10s_ld16_coif2_agf`** (SMAPE 21.908, 418K params)  
Only 0.6% worse than the winner with 5x fewer parameters.

### What to Test Next

1. **Test coif3 wavelet in TrendWaveletAELG sweep** -- the prior SOTA used coif3, not tested here
2. **Test eq_bcast basis dimension** -- prior SOTA used eq_bcast (bd=8 on Tourism), not tested here
3. **Combine agf + TrendWaveletAELG at ld16 with coif3** -- the sweep showed agf helps significantly, and the SOTA config uses ag0. An agf version of the SOTA config might beat it.
4. **GenericAE at 10 stacks with agf** (SMAPE 22.000 here) -- the resnet skip study showed GAE_10s at 20.526 with different training settings. Investigate what differs.

### Open Questions

- Why does the resnet skip study GenericAE (20.526) outperform this sweep's GenericAE (22.000) by 1.5 SMAPE? Training regime differences (epochs, LR, patience)?
- Is coif3 genuinely better than coif2/db3 for Tourism, or was the prior SOTA a lucky configuration?